# Add nD fields from NetCDF files to Extended IBTrACS

In [1]:
# Setup environment
import huracanpy
import xarray as xr
import numpy as np

from tqdm import tqdm

/Users/bourdin/Softs/miniforge3/envs/huracanpy/lib/python3.11/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [2]:
# Script parameters
## Extended IBTrACS file you want to add the attributes to (may be the minimal one or one you already added info to)
IN_FILE = "../files/extended_ibtracs_TS_1980-2022_reaching_BIWE.nc"
## NetCDF file to add
EOBS_FILE = "../../data/E-OBS/wind_percentiles.nc"
old_var_name, new_var_name = "fg", "wind_percentile" # In this specific case we want to change the variable name
## Output file automatically determined. You may change if you like.
OUT_FILE = IN_FILE[:-3]+"_with_EOBS-"+new_var_name+".nc"

In [3]:
# Open extended IBTrACS
eib = xr.open_dataset(IN_FILE)

In [4]:
# Open E-OBS file
EOBS_field = xr.open_dataset(EOBS_FILE)[old_var_name].rename(new_var_name)
EOBS_field = EOBS_field.sel(longitude=slice(-11.5,30), latitude = slice(35, 70))

In [5]:
# Select cyclone times from EOBS field
T = eib.time
EOBS_field_sample = EOBS_field.sel(time = T, method = "nearest") 
# NB: EOBS is daily whereas tracks are 3/6-hourly There will be repetitions of daily field. You can also first sub-sample Extended IBTrACS to daily to be more data-efficient.

In [6]:
# Merge the track and sst data
M = xr.merge([eib, EOBS_field_sample.drop_vars("time")],)

In [7]:
# Check the dataset
M

<xarray.Dataset> Size: 3GB
Dimensions:          (record: 14318, dataset: 4, longitude: 166, latitude: 140)
Coordinates:
  * dataset          (dataset) <U12 192B 'IBTrACS' ... 'TRACK-JRA3Q'
  * record           (record) int64 115kB 70560 70561 70562 ... 142129 142130
  * longitude        (longitude) float64 1kB -11.27 -11.02 ... 29.73 29.98
  * latitude         (latitude) float64 1kB 35.12 35.38 35.62 ... 69.62 69.88
Data variables: (12/21)
    track_id         (record) <U13 745kB ...
    lon              (dataset, record) float64 458kB ...
    lat              (dataset, record) float64 458kB ...
    name             (record) <U9 515kB ...
    pres             (dataset, record) float64 458kB ...
    wind10           (dataset, record) float64 458kB ...
    ...               ...
    usa_roci         (record) float64 115kB ...
    usa_rmw          (record) float64 115kB ...
    usa_eye          (record) float64 115kB ...
    usa_gust         (record) float64 115kB ...
    in_zone          (dataset, record) bool 57kB ...
    wind_percentile  (record, latitude, longitude) float64 3GB ...

In [ ]:
# Save
M.to_netcdf(OUT_FILE)